In [1]:
import numpy as np
import pandas as pd
import itertools
import matplotlib.pyplot as plt
from datetime import date
import re
from tqdm import tqdm
import time
# from pybedtools import BedTool
import random
import math
import pickle
import seaborn as sns
from scipy.stats import percentileofscore, pearsonr, rankdata
from scipy.spatial.distance import jaccard, euclidean, cosine
import functools
import operator
import gzip
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import sys
import os
from functools import partial
from contextlib import closing
from multiprocessing import Pool

today = date.today()

**Epilogos states**

**6/6/22, combining scores**

In [2]:
#6/6/22

#find the cosine simliarity for 1 region compared to all regions
def cosine_sim_per_ind(ooe_arr, ind):
    cosine_sim_arr = cosine_similarity(ooe_arr[[ind], :], ooe_arr)[0]
    cosine_sim_arr[ind] = -1
    cosine_sim_arr = (-np.sort(-cosine_sim_arr))[:100]
    return cosine_sim_arr

#find the cosine simliarity for 1 region compared to all regions in terms of region inds
def cosine_sim_regions_per_ind(ooe_arr, ind):
    cosine_sim_arr = cosine_similarity(ooe_arr[[ind], :], ooe_arr)[0]
    cosine_sim_arr[ind] = -1
    cosine_sim_inds_arr = (np.argsort(-cosine_sim_arr))[:100]
    return cosine_sim_inds_arr

In [3]:
num_cpus=1

In [5]:
#load the data
print('Loading data...', flush=True)
ooe_path = '/home/ctrader/region_similarity/data/regions_100k_CT20220602.DHSoee.npy'
ooe_arr = np.load(ooe_path)

#split on cluster and calculate cosine similarity for all regions
print('Calculating cosine similarity...', flush=True)
func1 = partial(cosine_sim_per_ind, ooe_arr)
func2 = partial(cosine_sim_regions_per_ind, ooe_arr)
with closing(Pool(num_cpus)) as pool:
#     top_100_cosine_vals = pool.map(func1, np.arange(len(ooe_arr)))
#     top_100_similar_inds = pool.map(func2, np.arange(len(ooe_arr)))
    top_100_cosine_valsDHS = pool.map(func1, np.arange(111))
    top_100_similar_indsDHS = pool.map(func2, np.arange(111))
pool.join()
top_100_cosine_valsDHS = np.array(top_100_cosine_valsDHS)
top_100_similar_indsDHS = np.array(top_100_similar_indsDHS)

top_100_rank_valsDHS = (len(ooe_arr) - (rankdata(-top_100_cosine_valsDHS, method='max', axis=1) - 1)) / len(ooe_arr)
top_100_rank_valsDHS = np.where(top_100_cosine_valsDHS!=0, top_100_rank_valsDHS, 0)

Loading data...
Calculating cosine similarity...


In [6]:
#load the data
print('Loading data...', flush=True)
ooe_path = '/home/ctrader/region_similarity/data/regions_100k_CT20220602.TFoee.npy'
ooe_arr = np.load(ooe_path)

#split on cluster and calculate cosine similarity for all regions
print('Calculating cosine similarity...', flush=True)
func1 = partial(cosine_sim_per_ind, ooe_arr)
func2 = partial(cosine_sim_regions_per_ind, ooe_arr)
with closing(Pool(num_cpus)) as pool:
#     top_100_cosine_vals = pool.map(func1, np.arange(len(ooe_arr)))
#     top_100_similar_inds = pool.map(func2, np.arange(len(ooe_arr)))
    top_100_cosine_valsTF = pool.map(func1, np.arange(111))
    top_100_similar_indsTF = pool.map(func2, np.arange(111))
pool.join()
top_100_cosine_valsTF = np.array(top_100_cosine_valsTF)
top_100_similar_indsTF = np.array(top_100_similar_indsTF)

top_100_rank_valsTF = (len(ooe_arr) - (rankdata(-top_100_cosine_valsTF, method='max', axis=1) - 1)) / len(ooe_arr)
top_100_rank_valsTF = np.where(top_100_cosine_valsTF!=0, top_100_rank_valsTF, 0)

Loading data...
Calculating cosine similarity...


In [11]:
region = 0

region_DHS_arr = np.zeros(len(ooe_arr))#change to len(top_100_similar_indsDHS) for full script so we don't have to load ooe_arr
region_DHS_arr[top_100_similar_indsDHS[region, :]] = top_100_rank_valsDHS[region, :]/100
region_TF_arr = np.zeros(len(ooe_arr))
region_TF_arr[top_100_similar_indsTF[region, :]] = top_100_rank_valsTF[region, :]/100

result = 1 - ((1 - region_TF_arr) * (1 - region_DHS_arr))
resulting_scores = np.append(1, -np.sort(-result)[:100])
resulting_regions = np.argsort(-result0)[:100]
# count_data0 = count_data[region, :3]
# count_data0 = np.vstack((count_data0, count_data[resulting0_regions, :3]))

In [107]:
f = open('/home/ctrader/region_similarity/data/Altius_100k_epilogos_hg19_WM20210518.sorted.bed', 'r')
regions = []
for line in f:
    regions.append(line.strip().split('\t'))
f.close()
regions = np.array(regions)

In [137]:
genome_coords = 'chr16:9000-9001'

chrm = genome_coords.split(':')[0]
start = int(genome_coords.split(':')[1].split('-')[0])
end = int(genome_coords.split(':')[1].split('-')[1])

region_ind = np.where((regions[:, 0]==chrm) & 
                      (regions[:, 1].astype(int)<=start) & 
                      (regions[:, 2].astype(int)>=end))[0]


if len(region_ind)!=1:
    sys.exit('Region not found')
else:
    f = open('/home/ctrader/region_similarity/data/test_CT20220606.txt', 'w')
    f.write(str(region_ind[0]))
    f.close

In [138]:
region_ind[0]

32432

In [119]:
regions[:, 1].astype(int)

array([     9000,     34000,     61000, ..., 155092800, 155117800,
       155142800])

In [140]:
f = open('/home/ctrader/region_similarity/data/region_of_interest_CT20220606.txt', 'r')
roi = int(f.readline())

In [141]:
roi

32432

In [25]:
#find the cosine simliarity for 1 region compared to all regions
def cosine_sim_per_ind(ooe_arr, sample_inds, tmp_path, ind):
    cosine_sim_arr = cosine_similarity(ooe_arr[[ind], :], ooe_arr)[0]
    if ind in sample_inds:
        np.save(os.path.join(tmp_path, 'sample_'+str(ind)+'_test_cosine_arr.npy'), cosine_sim_arr)
    cosine_sim_arr[ind] = -1######move to here
    cosine_sim_arr = (-np.sort(-cosine_sim_arr))[:100]
    return cosine_sim_arr

In [26]:
#find the cosine simliarity for 1 region compared to all regions in terms of region inds
def cosine_sim_regions_per_ind(ooe_arr, sample_inds, tmp_path, ind):
    cosine_sim_arr = cosine_similarity(ooe_arr[[ind], :], ooe_arr)[0]
    cosine_sim_arr[ind] = -1#####
    cosine_sim_inds_arr = (np.argsort(-cosine_sim_arr))[:100]
    return cosine_sim_inds_arr

In [27]:
#find the percentiles of top 100 cosine values for each region
def percentile_calc(cosine_samples, sample_top_cosine):
    #sample_top_cosine = top_100_cosine_vals[ind, :]
    sample_percentiles = []
    for score in sample_top_cosine:
        sample_percentiles.append(percentileofscore(cosine_samples, score))
    return sample_percentiles

In [37]:
#load the data
print('Loading data...', flush=True)

ooe_path = '/home/ctrader/region_similarity/data/regions_100k_CT20220602.DHSoee.npy'
ooe_arr = np.load(ooe_path)

sample_inds = np.array(random.sample(range(110), k=10))

#split on cluster and calculate cosine similarity for all regions
print('Calculating cosine similarity...', flush=True)
tmp_path = '/home/ctrader/region_similarity/data/DHS_tmp_files'
func1 = partial(cosine_sim_per_ind, ooe_arr, sample_inds, tmp_path)
func2 = partial(cosine_sim_regions_per_ind, ooe_arr, sample_inds, tmp_path)
with closing(Pool(1)) as pool:
    top_100_cosine_vals_DHS = pool.map(func1, np.arange(110))
    top_100_similar_inds_DHS = pool.map(func2, np.arange(110))
pool.join()
top_100_cosine_vals_DHS = np.array(top_100_cosine_vals_DHS)
top_100_similar_inds_DHS = np.array(top_100_similar_inds_DHS)

#sample 100 regions in order to calculate percentiles
print('Sampling...')
#sample_inds = np.array(random.sample(range(len(ooe_arr)), k=100))

cosine_samples = np.array([])
for i in sample_inds:
    cosine_samples = np.append(cosine_samples, np.load(os.path.join(tmp_path, 'sample_'+str(i)+'_test_cosine_arr.npy')))

print('Calculating percentiles...', flush=True)
func3 = partial(percentile_calc, cosine_samples)
with closing(Pool(1)) as pool:
    top_100_percentiles_DHS = pool.map(func3, top_100_cosine_vals_DHS[np.arange(110), :])
pool.join()
top_100_percentiles_DHS = np.array(top_100_percentiles_DHS)

Loading data...
Calculating cosine similarity...
Sampling...
Calculating percentiles...


In [36]:
#load the data
print('Loading data...', flush=True)

ooe_path = '/home/ctrader/region_similarity/data/regions_100k_CT20220602.TFoee.npy'
ooe_arr = np.load(ooe_path)

sample_inds = np.array(random.sample(range(110), k=10))

#split on cluster and calculate cosine similarity for all regions
print('Calculating cosine similarity...', flush=True)
tmp_path = '/home/ctrader/region_similarity/data/TF_tmp_files'
func1 = partial(cosine_sim_per_ind, ooe_arr, sample_inds, tmp_path)
func2 = partial(cosine_sim_regions_per_ind, ooe_arr, sample_inds, tmp_path)
with closing(Pool(1)) as pool:
    top_100_cosine_vals_TF = pool.map(func1, np.arange(110))
    top_100_similar_inds_TF = pool.map(func2, np.arange(110))
pool.join()
top_100_cosine_vals_TF = np.array(top_100_cosine_vals_TF)
top_100_similar_inds_TF = np.array(top_100_similar_inds_TF)

#sample 100 regions in order to calculate percentiles
print('Sampling...')
#sample_inds = np.array(random.sample(range(len(ooe_arr)), k=100))

cosine_samples = np.array([])
for i in sample_inds:
    cosine_samples = np.append(cosine_samples, np.load(os.path.join(tmp_path, 'sample_'+str(i)+'_test_cosine_arr.npy')))

print('Calculating percentiles...', flush=True)
func3 = partial(percentile_calc, cosine_samples)
with closing(Pool(1)) as pool:
    top_100_percentiles_TF = pool.map(func3, top_100_cosine_vals_TF[np.arange(110), :])
pool.join()
top_100_percentiles_TF = np.array(top_100_percentiles_TF)

Loading data...
Calculating cosine similarity...
Sampling...
Calculating percentiles...


In [117]:
region = 0

region0_DHS_arr = np.zeros(99996)
region0_DHS_arr[top_100_similar_inds_DHS[region, :]] = top_100_percentiles_DHS[region, :]/100
region0_TF_arr = np.zeros(99996)
region0_TF_arr[top_100_similar_inds_TF[region, :]] = top_100_percentiles_TF[region, :]/100

result0 = 1 - ((1 - region0_TF_arr) * (1 - region0_DHS_arr))
resulting0_scores = np.append(1, -np.sort(-result0)[:100])
resulting0_regions = np.argsort(-result0)[:100]
count_data0 = count_data[region, :3]
count_data0 = np.vstack((count_data0, count_data[resulting0_regions, :3]))

In [118]:
bed_path_name = os.path.join('/home/ctrader/public_html/', 'test_region0_top100_similar_regions_CT' + today.strftime("%Y%m%d") + '.bed')
f= open(bed_path_name, 'w')
for i in range(100):
    f.write(count_data0[i, 0] + '\t' + str(count_data0[i, 1]) + '\t' + str(count_data0[i, 2]) + '\t' + str(resulting0_scores[i]) + '\n')
f.close()
link_path = 'https://index.altius.org:3000/?application=viewer&roiURL=https%3A%2F%2Fencode%3Acollabor8%40resources.altius.org%2F~ctrader%2F' + 'test_region0_top100_similar_regions_CT' + today.strftime("%Y%m%d") + '.bed'
print(link_path)

https://index.altius.org:3000/?application=viewer&roiURL=https%3A%2F%2Fencode%3Acollabor8%40resources.altius.org%2F~ctrader%2Ftest_region0_top100_similar_regions_CT20220603.bed


In [123]:
print('https://epilogos.altius.org:3001/?application=viewer&sampleSet=vA&mode=single&genome=hg19&model=15&complexity=KL&group=all&chrLeft=chr1&chrRight=chr1&start=8000&stop=35000&roiURL=https%3A%2F%2Fencode%3Acollabor8%40resources.altius.org%2F~ctrader%2Ftest_region0_top100_similar_regions_CT20220603.bed')

https://epilogos.altius.org:3001/?application=viewer&sampleSet=vA&mode=single&genome=hg19&model=15&complexity=KL&group=all&chrLeft=chr1&chrRight=chr1&start=8000&stop=35000&roiURL=https%3A%2F%2Fencode%3Acollabor8%40resources.altius.org%2F~ctrader%2Ftest_region0_top100_similar_regions_CT20220603.bed


In [124]:
print('https://epilogos.altius.org:3001/?application=viewer&sampleSet=vA&mode=single&genome=hg19&model=15&complexity=KL&group=all&chrLeft=chr1&chrRight=chr1&start=8000&stop=35000&roiURL=https%3A%2F%2Fencode%3Acollabor8%40resources.altius.org%2F~ctrader%2Ftest_region50_top100_similar_regions_CT20220603.bed')

https://epilogos.altius.org:3001/?application=viewer&sampleSet=vA&mode=single&genome=hg19&model=15&complexity=KL&group=all&chrLeft=chr1&chrRight=chr1&start=8000&stop=35000&roiURL=https%3A%2F%2Fencode%3Acollabor8%40resources.altius.org%2F~ctrader%2Ftest_region50_top100_similar_regions_CT20220603.bed


In [125]:
print('https://epilogos.altius.org:3001/?application=viewer&sampleSet=vA&mode=single&genome=hg19&model=15&complexity=KL&group=all&chrLeft=chr1&chrRight=chr1&start=8000&stop=35000&roiURL=https%3A%2F%2Fencode%3Acollabor8%40resources.altius.org%2F~ctrader%2Ftest_region90_top100_similar_regions_CT20220603.bed')

https://epilogos.altius.org:3001/?application=viewer&sampleSet=vA&mode=single&genome=hg19&model=15&complexity=KL&group=all&chrLeft=chr1&chrRight=chr1&start=8000&stop=35000&roiURL=https%3A%2F%2Fencode%3Acollabor8%40resources.altius.org%2F~ctrader%2Ftest_region90_top100_similar_regions_CT20220603.bed


In [2]:
ooe_arr = np.load('/home/ctrader/region_similarity/data/regions_100k_CT20220602.DHSoee.npy')

In [3]:
def cosine_sim_per_ind(ooe_arr, ind):
    cosine_sim_arr = cosine_similarity(ooe_arr[[ind], :], ooe_arr)[0]
    return cosine_sim_arr

In [4]:
def percentile_calc(cosine_samples, top_100_cosine_vals, ind):
    sample_top_cosine = top_100_cosine_vals[ind, :]
    sample_percentiles = []
    for score in sample_top_cosine:
        sample_percentiles.append(percentileofscore(cosine_samples, score))
    return sample_percentiles

In [54]:
func1 = partial(cosine_sim_per_ind, ooe_arr)
with closing(Pool(1)) as pool:
    cosine_m = pool.map(func1, np.arange(110))
pool.join()

cosine_m = np.array(cosine_m)

In [91]:
cosine_m = cosine_m[1:, :]
cosine_m.shape

(0, 99996)

In [45]:
sample_inds = np.array(random.sample(range(110), k=10))
# sample_inds = np.array(random.sample(range(len(ooe_arr)), k=100))
cosine_samples = np.ndarray.flatten(cosine_m[sample_inds, :])

In [43]:
cosine_m[np.arange(110), np.arange(110)] = -1

top_100_cosine_vals = []
top_100_similar_inds = []

while len(cosine_m) > 0:
    row = cosine_m[0, :]
    top_100_cosine_vals.append((-np.sort(-row))[:100])
    top_100_similar_inds.append((np.argsort(-row))[:100])
    cosine_m = cosine_m[1:, :]
    
top_100_cosine_vals = np.array(top_100_cosine_vals)
top_100_similar_inds = np.array(top_100_similar_inds)

In [71]:
np.max(top_100_similar_inds - top_100_simlar_indstest)

0.0

In [47]:
# # cosine_m[np.arange(len(ooe_arr)), np.arange(len(ooe_arr))] = -1
# cosine_m[np.arange(110), np.arange(110)] = -1

# cosine_m[np.arange(110), np.arange(110)] = -1

# top_100_cosine_valstest = -np.sort(-cosine_m, axis=1)[:, :100]
# top_100_simlar_indstest = np.argsort(-cosine_m, axis=1)[:, :100]
# del cosine_m

In [78]:
func2 = partial(percentile_calc, cosine_samples, top_100_cosine_vals)
with closing(Pool(1)) as pool:
    top_100_percentiles = pool.map(func2, np.arange(110))
pool.join()

top_100_percentiles = np.array(top_100_percentiles)

In [121]:
count_data[0, :]

array(['chr1', '9000', '34000', 'AHR', '6', 'AIRE', '8', 'AP1_1', '29',
       'AP1_2', '51', 'ARI5A', '2', 'ARI5B', '3', 'BATF', '9', 'BCL6_1',
       '0', 'BCL6_2', '21', 'CCAAT_CEBP', '15', 'CENBP', '1', 'CPEB1',
       '3', 'CREB3_XBP1', '16', 'CREB_ATF_1', '21', 'CREB_ATF_2', '15',
       'CREB_ATF_3', '6', 'CTCF', '131', 'CUX_1', '4', 'CUX_2', '2',
       'CUX_3', '1', 'CUX_4', '2', 'DDIT3+CEBPA', '3', 'DMRT1', '6',
       'DMRT3', '1', 'E2F_1', '12', 'E2F_2', '145', 'E2F_3', '6', 'E2F_4',
       '35', 'EBF1', '36', 'Ebox_CACCTG', '117', 'Ebox_CACGTG_1', '57',
       'Ebox_CACGTG_2', '95', 'Ebox_CAGATGG', '143', 'Ebox_CAGCTG', '153',
       'Ebox_CATATG', '14', 'EGR', '60', 'ETS_1', '65', 'ETS_2', '200',
       'EVI1_MECOM', '0', 'EWSR1_FLI1', '10', 'FEZF1', '12', 'FOX_1', '8',
       'FOX_2', '1', 'FOX_3', '9', 'FOX_4', '44', 'FOX_5', '41', 'FOX_6',
       '4', 'FOX_7', '31', 'FOX_8', '16', 'FOX_9', '3', 'GATA', '21',
       'GCM', '22', 'GC-tract', '1242', 'GFI', '14', 'GLI', '

In [119]:
#load count data
filename = '/home/ctrader/region_similarity/data/regions_100k_CT20220602.TFmapped.bed'
count_data = []
f = open(filename, 'r')
for line in f:
    count_data.append(line.strip().split('\t'))
f.close()
count_data = np.array(count_data)

In [33]:
# seperating count labels and count values
count_val_inds = np.arange(4, len(count_data[0, :]), 2)
count_data_vals = count_data[:, count_val_inds].astype(int)
count_data_labels = count_data[0, count_val_inds-1]

In [34]:
#finding the total number of counts for each label
dir_path = '/home/ctrader/region_similarity/data/dhs_component_bed'
total_counts = np.array([], dtype=int)
for label in count_data_labels:
    label_path = os.path.join(dir_path, label+'.bed')
    f = open(label_path, 'r')
    total_counts = np.append(total_counts, len(f.readlines()))

In [35]:
#finding observed over expected for each label
expected = total_counts / len(count_data_vals)
ooe1 = count_data_vals/expected

In [3]:
t

,chromosome,start,end,TF motif cluster/group,match score,strand,TF motif,?
0,chr1,10005,10022,KLF/SP/2,3.7536,-,RREB1_MA0073.1,1
1,chr1,10011,10028,KLF/SP/2,3.7536,-,RREB1_MA0073.1,1
2,chr1,10017,10034,KLF/SP/2,3.7536,-,RREB1_MA0073.1,1
3,chr1,10023,10040,KLF/SP/2,3.7536,-,RREB1_MA0073.1,1
4,chr1,10029,10046,KLF/SP/2,3.7536,-,RREB1_MA0073.1,1
...,...,...,...,...,...,...,...,...
764808786,chrY_KI270740v1_random,37190,37196,SNAI2,7.7409,+,SNAI2_C2H2_1,2
764808787,chrY_KI270740v1_random,37190,37197,Ebox/CACCTG,6.6662,-,SNAI2_HUMAN.H11MO.0.A,2
764808788,chrY_KI270740v1_random,37191,37204,NR/11,6.7667,+,NR6A1_HUMAN.H11MO.0.B,1
764808789,chrY_KI270740v1_random,37219,37233,ZNF418,6.6390,-,ZN418_HUMAN.H11MO.0.C,1


In [2]:
motif_cosine = pd.read_pickle('/home/stsai/AugmentedGenomics/externalresources/data/motif_cosine_25kb.pkl')

In [3]:
motif_cosine

,0,1,2,3,4,5,6,7,8,9,...,99986,99987,99988,99989,99990,99991,99992,99993,99994,99995
0,1.000000,0.517479,0.431465,0.552631,0.757408,0.712969,0.791547,0.936062,0.0,0.298166,...,0.572527,0.537958,0.547472,0.633495,0.711792,0.575455,0.792505,0.499967,0.589655,0.567611
1,0.517479,1.000000,0.762757,0.763772,0.696884,0.623399,0.718753,0.594868,0.0,0.490550,...,0.750978,0.755607,0.753423,0.757235,0.661968,0.759152,0.678938,0.781730,0.767918,0.760462
2,0.431465,0.762757,1.000000,0.797089,0.711510,0.628755,0.604133,0.502751,0.0,0.549661,...,0.829889,0.827435,0.788805,0.805966,0.615844,0.737649,0.631472,0.795827,0.801242,0.835300
3,0.552631,0.763772,0.797089,1.000000,0.769799,0.747697,0.737495,0.618745,0.0,0.505982,...,0.842288,0.857433,0.818026,0.866355,0.679252,0.795590,0.679215,0.780998,0.839428,0.834777
4,0.757408,0.696884,0.711510,0.769799,1.000000,0.828851,0.725791,0.831603,0.0,0.479614,...,0.760627,0.746750,0.755000,0.766650,0.751872,0.710991,0.856740,0.721443,0.763542,0.765904
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99991,0.575455,0.759152,0.737649,0.795590,0.710991,0.684877,0.722019,0.639140,0.0,0.495989,...,0.800087,0.798722,0.854119,0.816975,0.827834,1.000000,0.736024,0.759348,0.835130,0.823254
99992,0.792505,0.678938,0.631472,0.679215,0.856740,0.752201,0.709921,0.859421,0.0,0.463185,...,0.691116,0.677775,0.713453,0.696293,0.825627,0.736024,1.000000,0.657401,0.710127,0.699917
99993,0.499967,0.781730,0.795827,0.780998,0.721443,0.681090,0.667499,0.570503,0.0,0.597014,...,0.818207,0.799005,0.850463,0.809266,0.633600,0.759348,0.657401,1.000000,0.837125,0.840804
99994,0.589655,0.767918,0.801242,0.839428,0.763542,0.757376,0.756817,0.653991,0.0,0.544632,...,0.865019,0.855989,0.848752,0.880025,0.678273,0.835130,0.710127,0.837125,1.000000,0.888681


In [4]:
motif_cosine = motif_cosine.to_numpy()
#setting cosine similarities of each region to itself as -1
motif_cosine[np.arange(len(motif_cosine)), np.arange(len(motif_cosine))]=-1
#find the top 100 similar region inds for each region
motif_cosine_top100inds = np.argsort(-motif_cosine, axis=1)[:, :100]
print('Found inds!')
#find the cosine similarity values for top 100 similar regions
motif_cosine_top100vals = -np.sort(-motif_cosine, axis=1)[:, :100]
print('Found values!')

Found inds!
Found values!


In [18]:
np.savez('/home/ctrader/region_similarity_data/motif_cosine_25kb_top100simregions_CT20220531.npz',
        top100inds=motif_cosine_top100inds, top100cosine=motif_cosine_top100vals)

In [ ]:
motif_cosine.to_csv('/home/ctrader/region_similarity_data/motif_cosine_25kb_CT20220531.tsv', sep='\t')

In [ ]:
del motif_cosine

In [ ]:
dhs_cosine = pd.read_pickle('/home/stsai/AugmentedGenomics/externalresources/data/dhs_cosine_25kb.pkl')

In [ ]:
dhs_cosine.to_csv('/home/ctrader/region_similarity_data/dhs_cosine_25kb_CT20220531.tsv', sep='\t')

In [ ]:
del dhs_cosine

In [ ]:
epilogos_cosine = pd.read_pickle('/home/stsai/AugmentedGenomics/externalresources/data/epilogos_cosine_25kb.pkl')

In [ ]:
epilogos_cosine.to_csv('/home/ctrader/region_similarity_data/epilogos_cosine_25kb_CT20220531.tsv', sep='\t')